# 📦 Amazon Parcel Defect Detection with YOLOv8-OBB (Train + Pretrained Evaluation)


In [ ]:
# --- 🔧 Step 1: Install YOLOv8 ---
!pip install ultralytics -q

In [ ]:
# --- 📥 Step 2: Clone the GitHub Dataset ---
!git clone https://github.com/saikisri97/Optimisation.git

In [ ]:
# --- 🛠️ Step 3: Patch data.yaml for local Colab paths ---
import os

yaml_path = "/content/Optimisation/For_AI_Lecture/data/Parcels-with-Defect-1/data.yaml"
os.makedirs(os.path.dirname(yaml_path), exist_ok=True)

with open(yaml_path, "w") as f:
    f.write("""train: /content/Optimisation/For_AI_Lecture/data/Parcels-with-Defect-1/train/images
val: /content/Optimisation/For_AI_Lecture/data/Parcels-with-Defect-1/train/images
test: /content/Optimisation/For_AI_Lecture/data/Parcels-with-Defect-1/train/images
nc: 2
names: ['defect parcel', 'no defect parcel']
""")

In [ ]:
# --- 🖼️ Step 4: Show Training Images with Labels (Both Classes) ---
import glob
import matplotlib.pyplot as plt
from PIL import Image
import os

image_dir = "/content/Optimisation/For_AI_Lecture/data/Parcels-with-Defect-1/train/images"
label_dir = "/content/Optimisation/For_AI_Lecture/data/Parcels-with-Defect-1/train/labels"

def get_class_label(label_file):
    with open(label_file, 'r') as f:
        lines = f.readlines()
    return int(lines[0].split()[0]) if lines else -1

# Categorize images by class 0 or 1
class0_imgs, class1_imgs = [], []
for img_file in sorted(glob.glob(f"{image_dir}/*.jpg")):
    base = os.path.basename(img_file).replace('.jpg', '.txt')
    label_file = os.path.join(label_dir, base)
    if os.path.exists(label_file):
        label = get_class_label(label_file)
        if label == 0 and len(class0_imgs) < 2:
            class0_imgs.append(img_file)
        elif label == 1 and len(class1_imgs) < 2:
            class1_imgs.append(img_file)
    if len(class0_imgs) >= 2 and len(class1_imgs) >= 2:
        break

# Display
plt.figure(figsize=(10, 5))
for i, img_path in enumerate(class0_imgs + class1_imgs):
    img = Image.open(img_path)
    label = "Defect" if i < 2 else "No Defect"
    plt.subplot(1, 4, i + 1)
    plt.imshow(img)
    plt.title(label)
    plt.axis("off")
plt.suptitle("🔍 Training Set: Defect vs No Defect")
plt.tight_layout()
plt.show()


In [ ]:
# --- 🚆 Step 5: Train for 1 Epoch (Demo) ---
from ultralytics import YOLO
model = YOLO("yolov8n-obb.pt")

In [ ]:
# ---- This step is just to show the training loop ----

model.train(
    data="/content/Optimisation/For_AI_Lecture/data/Parcels-with-Defect-1/data.yaml",
    epochs=1,
    imgsz=640,
    batch=8,
    name="parcel_defect_yolo_demo",
    workers=2
)

In [ ]:
# --- 🧠 Step 6: Load Pretrained Checkpoint (This is how the usage is done - For example ChatGPT  - we use the checkpoint on their Servers. ) ---
pretrained_url = "https://github.com/saikisri97/Optimisation/raw/master/For_AI_Lecture/data/yolov8n-obb-parcel-defect.pt"
checkpoint_path = "/content/yolov8n-obb-parcel-defect.pt"
!wget -O {checkpoint_path} {pretrained_url}

model = YOLO(checkpoint_path)


In [ ]:
# --- 📊 Step 7: Evaluate Pretrained Model ---
metrics = model.val(data="/content/Optimisation/For_AI_Lecture/data/Parcels-with-Defect-1/data.yaml", split='test')
print("Evaluation Metrics:", metrics)

In [ ]:
# --- 🔍 Step 8: Inference on Test Images and Visualize ---
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

results = model.predict(
    source="/content/Optimisation/For_AI_Lecture/data/Parcels-with-Defect-1/test/images",
    save=True,
    imgsz=640
)

# Show one prediction each for class 0 and class 1
found_0 = found_1 = False
output_dir = Path(results[0].save_dir)
for r, orig in zip(results, r.paths):
    labels = r.boxes.cls.int().tolist()
    if 0 in labels and not found_0:
        display_img_0 = r.save_dir / Path(orig).name
        found_0 = True
    if 1 in labels and not found_1:
        display_img_1 = r.save_dir / Path(orig).name
        found_1 = True
    if found_0 and found_1:
        break

plt.figure(figsize=(10, 4))
for i, path in enumerate([display_img_0, display_img_1]):
    img = cv2.imread(str(path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.subplot(1, 2, i + 1)
    plt.imshow(img)
    plt.title("Defect" if i == 0 else "No Defect")
    plt.axis("off")
plt.suptitle("📦 Predicted Output: Class-wise Inference")
plt.tight_layout()
plt.show()
